In [29]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from ollama import chat
import os
from datasets import Dataset
from ragas import evaluate
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall
from ragas.embeddings import LangchainEmbeddingsWrapper


Настройка API ключей

In [ ]:
HF_API_KEY = ""
os.environ["HF_TOKEN"] = HF_API_KEY

In [ ]:
file_path = r"/home/kagor/Загрузки/RAG/RAG_итог/RAG_инфа по распорядку и пр.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

Инициализация эмбеддингов

In [32]:
hf_embeddings_model = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23389.19it/s]
BertModel LOAD REPORT from: cointegrated/LaBSE-en-ru
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Чанки и Сплиттер

In [33]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_texts = []
for doc in docs:
    chunks = text_splitter.split_text(doc.page_content)
    all_texts.extend(chunks)

chunk_documents = [Document(page_content=text) for text in all_texts]

Создание или загрузка векторной базы Chroma

In [34]:
import os
import shutil
import tempfile

# тут временная папка с гарантированными правами
persist_directory = tempfile.mkdtemp()

print(f"папка для базы: {persist_directory}")

vector_db = Chroma.from_documents(
    documents=chunk_documents,
    embedding=hf_embeddings_model,
    persist_directory=persist_directory
)

папка для базы: /tmp/tmpp8pw6vq5


Настройка ретриверов и поиск контекста

In [35]:
my_text = "	Что будет, если зайти в корпус МИФИ по чужому пропуску? И можно ли провести с собой в университет друга/родителей, если у них нет пропуска?"

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunk_documents)
bm25_retriever.k = 10

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.0, 1]
)

retriever_results = ensemble_retriever.invoke(my_text)

Реранкер и формирование контекста из найденных чанков

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

pairs = [[my_text, doc.page_content] for doc in retriever_results]
rerank_scores = reranker.predict(pairs)

sorted_indices = sorted(range(len(rerank_scores)), key=lambda i: rerank_scores[i], reverse=True)
top_k = 5
reranked_docs = [retriever_results[i] for i in sorted_indices[:top_k]]

context = "\n\n".join([doc.page_content for doc in reranked_docs])

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 18955.40it/s]


Генерация ответа

In [37]:
response = chat(
    model='qwen2.5:1.5b-instruct',
    messages=[
        {
            "role": "system",
            "content": (
                "Ты - помощник студентам МИФИ. "
                "Отвечай только на основе контекста. "
                "Ты - помощник студентам МИФИ. Отвечай только на основе контекста."
                "Если данных достаточно — ответь кратко и по делу."
                "Если данных нет - напиши 'Информация отсутствует в документе'."
                "Не добавляй фразу об отсутствии информации, если ты уже что-то сказал."
            )
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Вопрос: {my_text}"
            )
        }
    ],
)

answer = response.message.content
print(answer)

1. **Случай № 1 (займитесь чужим пропуском):** Проходить по чужому пропуску запрещено и может повлечь наказание.
2. **Случай № 2 (пропустит друг/родителей по своему):** Запретный пропуск в университет не может быть использован, так как это нарушение пропускного режима. Друг или родители должны иметь официальный пропуск для доступа.

### Ответ:
1. **Проходить по чужому пропуску:** Это запрещено.
2. **Провести с другом/родителями по своему пропуску:** Нельзя, так как это нарушение пропускного режима.


Метрики

In [ ]:
llm_for_metrics = ChatOllama(model ='qwen2.5:1.5b-instruct')
embeddings_metrics = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru")